In [1]:
##### ==========================================
# CardioAgent-Ops
# Notebook 01: Data Preparation
# Phase 1 - Cell 1: Environment Setup
# ==========================================

import sys
import subprocess
import importlib.util
import importlib.metadata

# Required packages
packages = {
    "datasets": "datasets",
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn"
}

# Check and install missing packages only
for module, package in packages.items():

    if importlib.util.find_spec(module) is None:
        print(f"Installing {package}...")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package
        ])

    else:
        print(f"{package}: already installed")

# Import libraries
import pandas as pd
import numpy as np
from datasets import load_dataset

# Display versions
print("\n========== ENVIRONMENT ==========")

for package in packages.values():
    print(f"{package}: {importlib.metadata.version(package)}")

print("\nEnvironment setup completed successfully!")


datasets: already installed
pandas: already installed
numpy: already installed
scikit-learn: already installed

========== ENVIRONMENT ==========
datasets: 4.8.5
pandas: 2.3.3
numpy: 2.1.3
scikit-learn: 1.6.1

Environment setup completed successfully!


In [2]:
# ==========================================
# Phase 2 - Cell 2: Load MedQuAD
# ==========================================

from datasets import load_dataset

# Load MedQuAD from Hugging Face
medquad = load_dataset("lavita/MedQuAD")

# Display dataset structure
print("========== DATASET STRUCTURE ==========")
print(medquad)

# Display available splits
print("\n========== AVAILABLE SPLITS ==========")
print(medquad.keys())

# Inspect dataset features
for split_name, split_data in medquad.items():

    print(f"\n========== {split_name.upper()} ==========")
    print("Number of examples:", len(split_data))
    print("Columns:", split_data.column_names)
    print("Features:", split_data.features)

# Display one example
first_split = list(medquad.keys())[0]

print("\n========== FIRST EXAMPLE ==========")
print(medquad[first_split][0])


README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…): reconstructing file:   0%|          |  0.00B / 10.7MB            

data/train-00000-of-00001-e36383d177026d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

========== DATASET STRUCTURE ==========
DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 47441
    })
})

========== AVAILABLE SPLITS ==========
dict_keys(['train'])

========== TRAIN ==========
Number of examples: 47441
Columns: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']
Features: {'document_id': Value('string'), 'document_source': Value('string'), 'document_url': Value('string'), 'category': Value('string'), 'umls_cui': Value('string'), 'umls_semantic_types': Value('string'), 'umls_semantic_group': Value('string'), 'synonyms': Value('string'), 'question_id': Value('string'), 'question_focus': Value('s

In [3]:
# ==========================================
# Phase 3 - Cell 3: Dataset Exploration
# ==========================================

import pandas as pd

# Convert Hugging Face Dataset to Pandas
df = medquad["train"].to_pandas()

# 1. Dataset dimensions
print("========== DATASET OVERVIEW ==========")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# 2. Missing values
print("\n========== MISSING VALUES ==========")
missing = df.isnull().sum()
print(missing[missing > 0])

# 3. Medical categories
print("\n========== CATEGORY DISTRIBUTION ==========")
print(df["category"].value_counts(dropna=False).head(15))

# 4. Document sources
print("\n========== DOCUMENT SOURCES ==========")
print(df["document_source"].value_counts().head(15))

# 5. Question types
print("\n========== QUESTION TYPES ==========")
print(df["question_type"].value_counts(dropna=False))

# 6. Sample questions
print("\n========== SAMPLE QUESTIONS ==========")

for i, row in df.head(5).iterrows():
    print(f"\nQuestion {i + 1}: {row['question']}")
    print(f"Focus: {row['question_focus']}")
    print(f"Answer: {str(row['answer'])[:250]}...")


========== DATASET OVERVIEW ==========
Rows: 47441
Columns: 13

========== MISSING VALUES ==========
document_id                5
category               15431
umls_cui               16024
umls_semantic_types    16066
umls_semantic_group    16024
synonyms               22772
question_focus            14
answer                 31034
dtype: int64

========== CATEGORY DISTRIBUTION ==========
category
Disease    16256
None       15431
Drug       13681
Other       2073
Name: count, dtype: int64

========== DOCUMENT SOURCES ==========
document_source
ADAM                     17348
MPlusDrugs               12889
GHR                       5430
GARD                      5394
NIDDK                     1192
NINDS                     1088
MPlusHealthTopics          981
MPlusHerbsSupplements      792
NIHSeniorHealth            769
CancerGov                  729
NHLBI                      559
CDC                        270
Name: count, dtype: int64

========== QUESTION TYPES ==========
question_type


In [4]:
# =======================================
# Phase 3 - Cell 4: Missing Answers Analysis
# ========================================

# 1. Check missing and empty answers
missing_answers = df["answer"].isna().sum()

empty_answers = (
    df["answer"].fillna("").str.strip().eq("").sum()
)

print("========== ANSWER QUALITY ==========")
print("Missing answers:", missing_answers)
print("Empty answers:", empty_answers)

# 2. Missing answers by category
print("\n========== MISSING ANSWERS BY CATEGORY ==========")

category_analysis = df.groupby(
    "category", dropna=False
).agg(
    total_questions=("question", "count"),
    missing_answers=("answer", lambda x: x.isna().sum())
)

category_analysis["missing_percentage"] = (
    category_analysis["missing_answers"]
    / category_analysis["total_questions"] * 100
).round(2)

print(category_analysis)

# 3. Missing answers by document source
print("\n========== MISSING ANSWERS BY SOURCE ==========")

source_analysis = df.groupby(
    "document_source", dropna=False
).agg(
    total_questions=("question", "count"),
    missing_answers=("answer", lambda x: x.isna().sum())
)

source_analysis["missing_percentage"] = (
    source_analysis["missing_answers"]
    / source_analysis["total_questions"] * 100
).round(2)

print(source_analysis.sort_values(
    "missing_percentage", ascending=False
))

# 4. Inspect examples without answers
print("\n========== EXAMPLES WITHOUT ANSWERS ==========")

missing_samples = df[df["answer"].isna()].head(5)

for _, row in missing_samples.iterrows():
    print("\nDocument ID:", row["document_id"])
    print("Source:", row["document_source"])
    print("Category:", row["category"])
    print("Question focus:", row["question_focus"])
    print("Question:", row["question"])
    print("Answer:", row["answer"])

# 5. Inspect examples with valid answers
print("\n========== EXAMPLES WITH VALID ANSWERS ==========")

valid_samples = df[df["answer"].notna()].head(3)

for _, row in valid_samples.iterrows():
    print("\nQuestion:", row["question"])
    print("Answer:", str(row["answer"])[:200])


========== ANSWER QUALITY ==========
Missing answers: 31034
Empty answers: 31034

========== MISSING ANSWERS BY CATEGORY ==========
          total_questions  missing_answers  missing_percentage
category                                                      
Disease             16256            15570               95.78
Drug                13681            13681              100.00
Other                2073             1778               85.77
NaN                 15431                5                0.03

========== MISSING ANSWERS BY SOURCE ==========
                       total_questions  missing_answers  missing_percentage
document_source                                                            
ADAM                             17348            17348              100.00
MPlusDrugs                       12889            12889              100.00
MPlusHerbsSupplements              792              792              100.00
GARD                              5394                5      

In [5]:

# ==========================================
# Cell 5 - Cardiovascular Data Exploration
# ==========================================

# Focus on records with available answers
df_valid = df[df["answer"].notna()].copy()

# Explore the most frequent question focuses
print("========== TOP QUESTION FOCUSES ==========")

print(
    df_valid["question_focus"]
    .dropna()
    .value_counts()
    .head(40)
    .to_string()
)

# Explore categories among records with answers
print("\n========== AVAILABLE CATEGORIES ==========")

print(
    df_valid["category"]
    .value_counts(dropna=False)
    .to_string()
)

# Inspect questions containing cardiovascular terms
cardio_terms = [
    "heart", "cardiac", "cardiovascular",
    "blood pressure", "hypertension",
    "cholesterol", "artery", "vascular",
    "stroke", "arrhythmia", "coronary"
]

pattern = "|".join(cardio_terms)

cardio_candidates = df_valid[
    df_valid["question_focus"].fillna("").str.contains(
        pattern, case=False, regex=True
    )
    |
    df_valid["question"].fillna("").str.contains(
        pattern, case=False, regex=True
    )
]

print("\n========== INITIAL CARDIOVASCULAR CANDIDATES ==========")
print("Number of candidates:", len(cardio_candidates))

print(
    cardio_candidates[
        ["document_source", "category", "question_focus", "question"]
    ].head(30).to_string(index=False)
)


========== TOP QUESTION FOCUSES ==========
question_focus
Breast Cancer                           53
Prostate Cancer                         43
Stroke                                  35
Skin Cancer                             34
Alzheimer's Disease                     30
Colorectal Cancer                       29
Lung Cancer                             29
Heart Attack                            28
Heart Failure                           28
High Blood Cholesterol                  28
Causes of Diabetes                      28
High Blood Pressure                     27
Parkinson's Disease                     25
Leukemia                                22
Shingles                                21
Osteoporosis                            21
Hemochromatosis                         20
Age-related Macular Degeneration        20
Diabetes                                20
Psoriasis                               19
Diabetic Retinopathy                    19
Gum (Periodontal) Disease              

In [6]:

# ==========================================
# Cell 6 - Review Unique Cardiovascular Candidates
# ==========================================

# Group candidate questions by medical condition
focus_summary = (
    cardio_candidates
    .groupby("question_focus")
    .agg(
        number_of_questions=("question", "count"),
        sources=("document_source", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .sort_values("number_of_questions", ascending=False)
)

print("========== UNIQUE CANDIDATE CONDITIONS ==========")
print("Number of unique focuses:", len(focus_summary))

print("\n========== CANDIDATE CONDITIONS ==========")
print(focus_summary.to_string())

# Inspect potential false positives
print("\n========== POSSIBLE FALSE POSITIVES ==========")

possible_false_positives = focus_summary[
    focus_summary.index.str.contains(
        "vascular", case=False, na=False
    )
]

print(possible_false_positives.to_string())


========== UNIQUE CANDIDATE CONDITIONS ==========
Number of unique focuses: 75

========== CANDIDATE CONDITIONS ==========
                                                                                               number_of_questions                                           sources
question_focus                                                                                                                                                      
Stroke                                                                                                          35  MPlusHealthTopics, NHLBI, NIHSeniorHealth, NINDS
Heart Failure                                                                                                   28         MPlusHealthTopics, NHLBI, NIHSeniorHealth
High Blood Cholesterol                                                                                          28                            NHLBI, NIHSeniorHealth
Heart Attack                                        

In [7]:

# ==========================================
# CardioAgent-Ops
# Notebook 01: Data Preparation
# Cell 7 - Cardiovascular Focus Selection
# ==========================================

# 1. Define the cardiovascular topics to include
include_focus = [
    "stroke",
    "heart failure",
    "heart attack",
    "high blood cholesterol",
    "high blood pressure",
    "diabetes, heart disease, and stroke",
    "renal artery stenosis",
    "diabetic heart disease",
    "carotid artery disease",
    "broken heart syndrome",
    "arrhythmia",
    "peripheral artery disease",
    "high blood pressure and kidney disease",
    "pulmonary hypertension",
    "sudden cardiac arrest",
    "heart disease in women",
    "heart palpitations",
    "coronary heart disease",
    "coronary microvascular disease",
    "heart valve disease",
    "heart block",
    "congenital heart defects",
    "prevent diabetes problems: keep your heart and blood vessels healthy",
    "progressive familial heart block",
    "pulmonary arterial hypertension",
    "holes in the heart",
    "critical congenital heart disease",
    "heart murmur",
    "hypercholesterolemia",
    "atrial fibrillation and stroke",
    "familial hypercholesterolemia",
    "coronary heart disease risk factors",
    "hypoplastic left heart syndrome",
    "familial progressive cardiac conduction defect",
    "cardiac valvular dysplasia, x-linked",
    "cardiac rehabilitation",
    "aberrant subclavian artery",
    "cardiac arrest",
    "heart valve diseases",
    "heart diseases",
    "coronary artery disease",
    "cholesterol",
    "conotruncal heart malformations",
    "coronary artery bypass surgery",
    "heart transplantation",
    "heart surgery",
    "heart diseases--prevention",
    "hemorrhagic stroke",
    "ischemic stroke",
    "high blood pressure in pregnancy",
    "low blood pressure",
    "hypercholesterolemia, autosomal dominant",
    "hypoplastic right heart syndrome",
    "smoking and your heart",
    "progressive familial heart block type 1a",
    "stroke rehabilitation",
    "blood pressure medicines"
]

# 2. Normalize focus names to avoid case differences
candidate_focus = cardio_candidates["question_focus"].fillna("").str.strip().str.lower()

# 3. Select the approved topics
cardio_selected = cardio_candidates[
    candidate_focus.isin(include_focus)
].copy()

# 4. Display the selection summary
selected_summary = (
    cardio_selected
    .groupby("question_focus")
    .agg(
        number_of_questions=("question", "count"),
        sources=("document_source", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .sort_values("number_of_questions", ascending=False)
)

print("========== CARDIOVASCULAR SELECTION ==========")
print("Selected questions:", len(cardio_selected))
print("Selected unique focuses:", len(selected_summary))

print("\n========== SELECTED FOCUSES ==========")
print(selected_summary.to_string())

# 5. Show candidate focuses not yet selected
not_selected = sorted(
    set(cardio_candidates["question_focus"].dropna())
    - set(selected_summary.index)
)

print("\n========== NOT SELECTED - REVIEW LATER ==========")
for focus in not_selected:
    print("-", focus)


========== CARDIOVASCULAR SELECTION ==========
Selected questions: 352
Selected unique focuses: 58

========== SELECTED FOCUSES ==========
                                                                      number_of_questions                                           sources
question_focus                                                                                                                             
Stroke                                                                                 35  MPlusHealthTopics, NHLBI, NIHSeniorHealth, NINDS
Heart Attack                                                                           28         MPlusHealthTopics, NHLBI, NIHSeniorHealth
Heart Failure                                                                          28         MPlusHealthTopics, NHLBI, NIHSeniorHealth
High Blood Cholesterol                                                                 28                            NHLBI, NIHSeniorHealth
High Blood Pressure  

In [8]:

# ==========================================
# CardioAgent-Ops
# Notebook 01: Data Preparation
# Cell 8 - Review Excluded Candidates
# ==========================================

review_focus = [
    "Celiac artery compression syndrome",
    "Ehlers-Danlos syndrome, vascular type",
    "Disseminated Intravascular Coagulation",
    "Hereditary endotheliopathy, retinopathy, nephropathy, and stroke",
    "Heart-hand syndrome, Slovenian type",
    "Heart-hand syndrome, Spanish type",
    "Hirschsprung disease polydactyly heart disease",
    "mitochondrial encephalomyopathy, lactic acidosis, and stroke-like episodes",
    "Familial avascular necrosis of the femoral head",
    "Hereditary vascular retinopathy",
    "Vascular Diseases"
]

review_df = cardio_candidates[
    cardio_candidates["question_focus"].isin(review_focus)
]

print("========== EXCLUDED TOPICS REVIEW ==========")

for focus, group in review_df.groupby("question_focus"):
    print("\n" + "=" * 65)
    print("FOCUS:", focus)
    print("Questions:", len(group))

    for _, row in group.head(2).iterrows():
        print("\nQuestion:", row["question"])
        print("Answer:", str(row["answer"])[:350])


========== EXCLUDED TOPICS REVIEW ==========

FOCUS: Celiac artery compression syndrome
Questions: 5

Question: What is (are) Celiac artery compression syndrome ?
Answer: Celiac artery compression syndrome is a rare disorder characterized by chronic, recurrent abdominal pain related to compression of the celiac artery (which supplies blood to the upper abdominal organs) by the median arcuate ligament (a muscular fibrous band of the diaphragm). It usually presents with symptoms of abdominal pain, weight loss, and an 

Question: What are the symptoms of Celiac artery compression syndrome ?
Answer: What are the signs and symptoms of celiac artery compression syndrome?  Classically, individuals with celiac artery compression syndrome present with a triad of abdominal pain after eating, weight loss (usually >20 pounds), and abdominal bruit (abnormal sound of a blood vessel when blocked or narrowed).  One review found that abdominal pain is the 

FOCUS: Disseminated Intravascular Coagulation

In [9]:

# ==========================================
# CardioAgent-Ops
# Notebook 01: Data Preparation
# Cell 9 - Final Ambiguous Topics Review
# ==========================================

final_review_focus = [
    "Ehlers-Danlos syndrome, vascular type",
    "Hereditary endotheliopathy, retinopathy, nephropathy, and stroke",
    "Heart-hand syndrome, Slovenian type",
    "Heart-hand syndrome, Spanish type",
    "Hirschsprung disease polydactyly heart disease",
    "Vascular Diseases"
]

final_review = cardio_candidates[
    cardio_candidates["question_focus"].isin(final_review_focus)
]

for focus, group in final_review.groupby("question_focus"):
    print("\n" + "=" * 70)
    print("FOCUS:", focus)

    for _, row in group.iterrows():
        print("\nQuestion:", row["question"])
        print("\nAnswer:\n", row["answer"])



FOCUS: Ehlers-Danlos syndrome, vascular type

Question: What is (are) Ehlers-Danlos syndrome, vascular type ?

Answer:
 Ehlers-Danlos syndrome (EDS), vascular type is an inherited connective tissue disorder that is caused by defects in a protein called collagen. It is generally considered the most severe form of Ehlers-Danlos syndrome. Common symptoms include thin, translucent skin; easy bruising; characteristic facial appearance; and fragile arteries, muscles and internal organs. EDS, vascular type is caused by changes (mutations) in the COL3A1 gene and is inherited in an autosomal dominant manner. Treatment and management is focused on preventing serious complications and relieving associated signs and symptoms.

Question: What are the symptoms of Ehlers-Danlos syndrome, vascular type ?

Answer:
 What are the signs and symptoms of Ehlers-Danlos syndrome, vascular type? The signs and symptoms of Ehlers-Danlos syndrome (EDS), vascular type vary but may include: Fragile tissues (includ

In [10]:

# ==========================================
# CardioAgent-Ops
# Notebook 01: Data Preparation
# Cell 10 - Final Cardiovascular Selection
# ==========================================

# 1. Additional cardiovascular-related topics
supplementary_focus = [
    "Ehlers-Danlos syndrome, vascular type",
    "Heart-hand syndrome, Slovenian type",
    "Heart-hand syndrome, Spanish type",
    "Hereditary endotheliopathy, retinopathy, nephropathy, and stroke",
    "Hirschsprung disease polydactyly heart disease",
    "Vascular Diseases"
]

# 2. Select supplementary records
supplementary_data = cardio_candidates[
    cardio_candidates["question_focus"].isin(supplementary_focus)
].copy()

# 3. Combine main and supplementary data
cardio_final = pd.concat(
    [cardio_selected, supplementary_data],
    ignore_index=True
)

# 4. Add a group label
cardio_final["topic_group"] = "main"

cardio_final.loc[
    cardio_final["question_focus"].isin(supplementary_focus),
    "topic_group"
] = "supplementary"

# 5. Display final selection statistics
print("========== FINAL CARDIOVASCULAR DATASET ==========")

print("Total questions:", len(cardio_final))

print("\nQuestions by topic group:")
print(cardio_final["topic_group"].value_counts())

print("\nUnique question focuses:")
print(cardio_final["question_focus"].nunique())

print("\nQuestions by source:")
print(cardio_final["document_source"].value_counts())

print("\nMissing answers:")
print(cardio_final["answer"].isna().sum())

print("\nDuplicate questions:")
print(cardio_final["question"].duplicated().sum())


========== FINAL CARDIOVASCULAR DATASET ==========
Total questions: 366

Questions by topic group:
topic_group
main             352
supplementary     14
Name: count, dtype: int64

Unique question focuses:
64

Questions by source:
document_source
NHLBI                139
NIHSeniorHealth      105
NIDDK                 36
GARD                  31
MPlusHealthTopics     27
GHR                   20
NINDS                  8
Name: count, dtype: int64

Missing answers:
0

Duplicate questions:
128


In [11]:

# ==========================================
# Cell 11 - Duplicate Analysis
# ==========================================

# 1. Extract duplicated questions
duplicates = cardio_final[
    cardio_final["question"].duplicated(keep=False)
].copy()

# 2. Group duplicates by question
duplicate_summary = (
    duplicates
    .groupby("question")
    .agg(
        occurrences=("question", "count"),
        number_of_sources=("document_source", "nunique"),
        sources=("document_source", lambda x: ", ".join(sorted(x.dropna().unique()))),
        focuses=("question_focus", lambda x: ", ".join(sorted(x.dropna().unique()))),
        unique_answers=("answer", "nunique")
    )
    .sort_values("occurrences", ascending=False)
)

print("========== DUPLICATE ANALYSIS ==========")

print("Rows involved in duplicates:", len(duplicates))
print("Unique duplicated questions:", len(duplicate_summary))

print("\n========== DUPLICATE TYPES ==========")

print(
    duplicate_summary["unique_answers"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n========== TOP 15 DUPLICATED QUESTIONS ==========")

print(duplicate_summary.head(15).to_string())

# 3. Inspect examples with different answers
print("\n========== EXAMPLES WITH DIFFERENT ANSWERS ==========")

different_answer_questions = duplicate_summary[
    duplicate_summary["unique_answers"] > 1
].head(3).index

for question in different_answer_questions:
    print("\nQuestion:", question)

    sample = duplicates[duplicates["question"] == question]

    for _, row in sample.iterrows():
        print("\nSource:", row["document_source"])
        print("Focus:", row["question_focus"])
        print("Answer:", str(row["answer"])[:300])


========== DUPLICATE ANALYSIS ==========
Rows involved in duplicates: 174
Unique duplicated questions: 46

========== DUPLICATE TYPES ==========
unique_answers
1      1
2     20
3     10
4      6
5      3
6      2
7      1
11     1
13     1
19     1

========== TOP 15 DUPLICATED QUESTIONS ==========
                                                     occurrences  number_of_sources                                           sources                              focuses  unique_answers
question                                                                                                                                                                                  
What is (are) High Blood Cholesterol ?                        19                  2                            NHLBI, NIHSeniorHealth               High Blood Cholesterol              19
What is (are) Stroke ?                                        13                  4  MPlusHealthTopics, NHLBI, NIHSeniorHealth, NINDS     

In [12]:
# ==========================================
# Cell 12 - Exact Duplicate Removal
# ==========================================

# Keep one occurrence of each exact question-answer pair
cardio_deduplicated = cardio_final.drop_duplicates(
    subset=["question", "answer"],
    keep="first"
).copy()

# Reset dataframe index
cardio_deduplicated.reset_index(drop=True, inplace=True)

print("========== DEDUPLICATION RESULTS ==========")

print("Original rows:", len(cardio_final))
print("Rows after deduplication:", len(cardio_deduplicated))
print("Exact duplicates removed:", len(cardio_final) - len(cardio_deduplicated))

print("\n========== DATA QUALITY CHECK ==========")

print(
    "Remaining exact duplicates:",
    cardio_deduplicated.duplicated(
        subset=["question", "answer"]
    ).sum()
)

print(
    "Missing questions:",
    cardio_deduplicated["question"].isna().sum()
)

print(
    "Missing answers:",
    cardio_deduplicated["answer"].isna().sum()
)

print("\n========== TOPIC DISTRIBUTION ==========")

print(
    cardio_deduplicated["topic_group"]
    .value_counts()
    .to_string()
)

========== DEDUPLICATION RESULTS ==========
Original rows: 366
Rows after deduplication: 364
Exact duplicates removed: 2

========== DATA QUALITY CHECK ==========
Remaining exact duplicates: 0
Missing questions: 0
Missing answers: 0

========== TOPIC DISTRIBUTION ==========
topic_group
main             350
supplementary     14


In [13]:
# ==========================================
# Cell 13 - Text Cleaning
# ==========================================

import re

# Create a copy to preserve the previous dataset
cardio_cleaned = cardio_deduplicated.copy()

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Remove invisible characters
    text = text.replace("\ufeff", " ")
    text = text.replace("\u200b", " ")

    # Normalize whitespace and line breaks
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Clean questions and answers
cardio_cleaned["question"] = cardio_cleaned["question"].apply(clean_text)
cardio_cleaned["answer"] = cardio_cleaned["answer"].apply(clean_text)

# Remove rows with empty questions or answers
cardio_cleaned = cardio_cleaned[
    (cardio_cleaned["question"] != "") &
    (cardio_cleaned["answer"] != "")
].copy()

cardio_cleaned.reset_index(drop=True, inplace=True)


# Display results
print("========== TEXT CLEANING RESULTS ==========")

print("Rows before cleaning:", len(cardio_deduplicated))
print("Rows after cleaning:", len(cardio_cleaned))
print("Rows removed:", len(cardio_deduplicated) - len(cardio_cleaned))

print("\n========== VALIDATION ==========")

print("Empty questions:", (cardio_cleaned["question"] == "").sum())
print("Empty answers:", (cardio_cleaned["answer"] == "").sum())

print(
    "Remaining exact duplicates:",
    cardio_cleaned.duplicated(subset=["question", "answer"]).sum()
)

print("\n========== CLEANED EXAMPLES ==========")

for _, row in cardio_cleaned.head(3).iterrows():
    print("\nQuestion:", row["question"][:200])
    print("Answer:", row["answer"][:300])

========== TEXT CLEANING RESULTS ==========
Rows before cleaning: 364
Rows after cleaning: 364
Rows removed: 0

========== VALIDATION ==========
Empty questions: 0
Empty answers: 0
Remaining exact duplicates: 0

========== CLEANED EXAMPLES ==========

Question: What is (are) pulmonary arterial hypertension ?
Answer: Pulmonary arterial hypertension is a progressive disorder characterized by abnormally high blood pressure (hypertension) in the pulmonary artery, the blood vessel that carries blood from the heart to the lungs. Pulmonary arterial hypertension is one form of a broader condition known as pulmonary hyp

Question: How many people are affected by pulmonary arterial hypertension ?
Answer: In the United States, about 1,000 new cases of pulmonary arterial hypertension are diagnosed each year. This disorder is twice as common in females as in males.

Question: What are the genetic changes related to pulmonary arterial hypertension ?
Answer: Mutations in the BMPR2 gene are the most c

In [14]:
# ==========================================
# CardioAgent-Ops
# Notebook 01: Data Preparation
# Cell 14 - Text Length Analysis
# ==========================================

# Calculate text lengths
cardio_cleaned["question_length"] = (
    cardio_cleaned["question"].apply(lambda x: len(x.split()))
)

cardio_cleaned["answer_length"] = (
    cardio_cleaned["answer"].apply(lambda x: len(x.split()))
)

print("========== DATASET LENGTH ANALYSIS ==========")

print("\nQuestion length (words):")
print(cardio_cleaned["question_length"].describe())

print("\nAnswer length (words):")
print(cardio_cleaned["answer_length"].describe())

print("\n========== LONG ANSWERS ==========")

print(
    cardio_cleaned.nlargest(10, "answer_length")[
        ["question_focus", "question_length", "answer_length"]
    ].to_string(index=False)
)

print("\n========== LENGTH THRESHOLDS ==========")

for threshold in [256, 512, 1024, 2048]:
    count = (cardio_cleaned["answer_length"] > threshold).sum()
    percentage = count / len(cardio_cleaned) * 100

    print(
        f"Answers > {threshold} words: "
        f"{count} ({percentage:.2f}%)"
    )

========== DATASET LENGTH ANALYSIS ==========

Question length (words):
count    364.000000
mean       7.719780
std        1.915155
min        4.000000
25%        6.000000
50%        7.000000
75%        9.000000
max       16.000000
Name: question_length, dtype: float64

Answer length (words):
count     364.000000
mean      355.931319
std       388.117088
min        10.000000
25%       115.750000
50%       217.500000
75%       468.750000
max      2406.000000
Name: answer_length, dtype: float64

========== LONG ANSWERS ==========
                     question_focus  question_length  answer_length
             Heart Disease in Women               10           2406
                Heart Valve Disease                9           2272
                             Stroke                7           1988
                       Heart Attack                8           1919
             Coronary Heart Disease                9           1821
                      Heart Failure                8      

In [15]:
# ==========================================
# Cell 15 - Token Length Analysis
# ==========================================

!pip install -q transformers

from transformers import AutoTokenizer

# Target model
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Load tokenizer only
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Measure token lengths
def count_tokens(text):
    return len(
        tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]
    )

cardio_cleaned["question_tokens"] = (
    cardio_cleaned["question"].apply(count_tokens)
)

cardio_cleaned["answer_tokens"] = (
    cardio_cleaned["answer"].apply(count_tokens)
)

# Combined length (question + answer)
cardio_cleaned["total_tokens"] = (
    cardio_cleaned["question_tokens"] +
    cardio_cleaned["answer_tokens"]
)

print("========== TOKEN LENGTH ANALYSIS ==========")

print("\nQuestion tokens:")
print(cardio_cleaned["question_tokens"].describe())

print("\nAnswer tokens:")
print(cardio_cleaned["answer_tokens"].describe())

print("\nTotal tokens:")
print(cardio_cleaned["total_tokens"].describe())

print("\n========== TOKEN LENGTH THRESHOLDS ==========")

for threshold in [512, 1024, 1536, 2048, 3072, 4096]:
    count = (cardio_cleaned["total_tokens"] > threshold).sum()
    percentage = count / len(cardio_cleaned) * 100

    print(
        f"Examples > {threshold} tokens: "
        f"{count} ({percentage:.2f}%)"
    )

print("\n========== LONGEST EXAMPLES ==========")

print(
    cardio_cleaned.nlargest(10, "total_tokens")[
        ["question_focus", "question_tokens",
         "answer_tokens", "total_tokens"]
    ].to_string(index=False)
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

========== TOKEN LENGTH ANALYSIS ==========

Question tokens:
count    364.000000
mean       9.730769
std        2.822390
min        4.000000
25%        8.000000
50%        9.000000
75%       11.000000
max       22.000000
Name: question_tokens, dtype: float64

Answer tokens:
count     364.000000
mean      450.057692
std       492.682280
min        17.000000
25%       148.000000
50%       279.000000
75%       594.750000
max      3091.000000
Name: answer_tokens, dtype: float64

Total tokens:
count     364.000000
mean      459.788462
std       492.423245
min        27.000000
25%       160.000000
50%       287.500000
75%       607.000000
max      3102.000000
Name: total_tokens, dtype: float64

========== TOKEN LENGTH THRESHOLDS ==========
Examples > 512 tokens: 106 (29.12%)
Examples > 1024 tokens: 32 (8.79%)
Examples > 1536 tokens: 17 (4.67%)
Examples > 2048 tokens: 11 (3.02%)
Examples > 3072 tokens: 1 (0.27%)
Examples > 4096 tokens: 0 (0.00%)

========== LONGEST EXAMPLES ==========
      

In [16]:
# ==========================================
# Cell 16 - Qwen Conversation Formatting
# ==========================================

import json

SYSTEM_PROMPT = (
    "You are a medical information assistant specialized in "
    "cardiovascular topics. Provide clear, informative answers "
    "based on the supplied medical information. Do not invent "
    "medical facts or replace professional medical advice."
)

# Build conversation examples
cardio_conversations = []

for _, row in cardio_cleaned.iterrows():

    conversation = {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": row["question"]
            },
            {
                "role": "assistant",
                "content": row["answer"]
            }
        ],
        "metadata": {
            "question_focus": row["question_focus"],
            "document_source": row["document_source"],
            "document_id": row["document_id"],
            "topic_group": row["topic_group"]
        }
    }

    cardio_conversations.append(conversation)


# Display dataset statistics
print("========== CONVERSATION FORMATTING ==========")

print("Total conversations:", len(cardio_conversations))

print("\n========== FIRST CONVERSATION ==========")

print(
    json.dumps(
        cardio_conversations[0],
        indent=4,
        ensure_ascii=False
    )[:2000]
)

print("\n========== STRUCTURE VALIDATION ==========")

valid_roles = {"system", "user", "assistant"}

valid_examples = all(
    len(item["messages"]) == 3
    and all(msg["role"] in valid_roles for msg in item["messages"])
    and all(isinstance(msg["content"], str) for msg in item["messages"])
    for item in cardio_conversations
)

print("All conversations valid:", valid_examples)
print("Messages per conversation:", len(cardio_conversations[0]["messages"]))
print("Metadata preserved:", "metadata" in cardio_conversations[0])

========== CONVERSATION FORMATTING ==========
Total conversations: 364

========== FIRST CONVERSATION ==========
{
    "messages": [
        {
            "role": "system",
            "content": "You are a medical information assistant specialized in cardiovascular topics. Provide clear, informative answers based on the supplied medical information. Do not invent medical facts or replace professional medical advice."
        },
        {
            "role": "user",
            "content": "What is (are) pulmonary arterial hypertension ?"
        },
        {
            "role": "assistant",
            "content": "Pulmonary arterial hypertension is a progressive disorder characterized by abnormally high blood pressure (hypertension) in the pulmonary artery, the blood vessel that carries blood from the heart to the lungs. Pulmonary arterial hypertension is one form of a broader condition known as pulmonary hypertension. Pulmonary hypertension occurs when most of the very small arteries 

In [17]:
# ==========================================
# Cell 17 - Split Group Analysis
# ==========================================

# Analyze question_focus groups
focus_groups = (
    cardio_cleaned
    .groupby("question_focus", dropna=False)
    .agg(
        number_of_questions=("question", "count"),
        number_of_sources=("document_source", "nunique")
    )
    .sort_values("number_of_questions", ascending=False)
)

print("========== GROUP ANALYSIS ==========")

print("Total conversations:", len(cardio_cleaned))
print("Unique question_focus groups:", cardio_cleaned["question_focus"].nunique())
print("Missing question_focus:", cardio_cleaned["question_focus"].isna().sum())

print("\n========== GROUP SIZE DISTRIBUTION ==========")

print(focus_groups["number_of_questions"].describe())

print("\n========== TOP 15 GROUPS ==========")

print(focus_groups.head(15).to_string())

print("\n========== SMALLEST GROUPS ==========")

print(focus_groups.tail(10).to_string())

========== GROUP ANALYSIS ==========
Total conversations: 364
Unique question_focus groups: 64
Missing question_focus: 0

========== GROUP SIZE DISTRIBUTION ==========
count    64.000000
mean      5.687500
std       7.478806
min       1.000000
25%       1.000000
50%       4.000000
75%       7.000000
max      35.000000
Name: number_of_questions, dtype: float64

========== TOP 15 GROUPS ==========
                                     number_of_questions  number_of_sources
question_focus                                                             
Stroke                                                35                  4
Heart Failure                                         28                  3
High Blood Cholesterol                                28                  2
Heart Attack                                          28                  3
High Blood Pressure                                   27                  3
Diabetes, Heart Disease, and Stroke                   11             

In [18]:
# ==========================================
# Cell 18 - Grouped Dataset Split
# ==========================================

from sklearn.model_selection import GroupShuffleSplit

# Prepare data
X = cardio_cleaned.copy()
groups = X["question_focus"]

# First split: 85% development, 15% test
split_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_val_idx, test_idx = next(
    split_1.split(X, groups=groups)
)

train_val = X.iloc[train_val_idx].copy()
test_data = X.iloc[test_idx].copy()

# Second split: approximately 70% train, 15% validation
split_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.1765,
    random_state=42
)

train_idx, val_idx = next(
    split_2.split(
        train_val,
        groups=train_val["question_focus"]
    )
)

train_data = train_val.iloc[train_idx].copy()
val_data = train_val.iloc[val_idx].copy()

# Reset indexes
train_data.reset_index(drop=True, inplace=True)
val_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)

# Display split statistics
print("========== DATASET SPLIT RESULTS ==========")

for name, dataset in [
    ("TRAIN", train_data),
    ("VALIDATION", val_data),
    ("TEST", test_data)
]:
    print(f"\n{name}")
    print("Examples:", len(dataset))
    print("Percentage:", round(len(dataset) / len(X) * 100, 2), "%")
    print("Unique topics:", dataset["question_focus"].nunique())

# Check group leakage
train_groups = set(train_data["question_focus"])
val_groups = set(val_data["question_focus"])
test_groups = set(test_data["question_focus"])

print("\n========== DATA LEAKAGE VALIDATION ==========")

print("Train / Validation overlap:", len(train_groups & val_groups))
print("Train / Test overlap:", len(train_groups & test_groups))
print("Validation / Test overlap:", len(val_groups & test_groups))

print("\n========== FINAL VALIDATION ==========")

print(
    "Total examples:",
    len(train_data) + len(val_data) + len(test_data)
)

print(
    "All groups separated:",
    len(train_groups & val_groups) == 0
    and len(train_groups & test_groups) == 0
    and len(val_groups & test_groups) == 0
)

========== DATASET SPLIT RESULTS ==========

TRAIN
Examples: 243
Percentage: 66.76 %
Unique topics: 44

VALIDATION
Examples: 89
Percentage: 24.45 %
Unique topics: 10

TEST
Examples: 32
Percentage: 8.79 %
Unique topics: 10

========== DATA LEAKAGE VALIDATION ==========
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0

========== FINAL VALIDATION ==========
Total examples: 364
All groups separated: True


In [19]:
# ==========================================
# Cell 19 - Optimized Grouped Split
# ==========================================

from sklearn.model_selection import GroupShuffleSplit

X = cardio_cleaned.copy()
groups = X["question_focus"]

target = [0.70, 0.15, 0.15]

best_score = float("inf")
best_split = None

# Search for a better group distribution
for seed in range(2000):

    # First split: development and test
    splitter_1 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15,
        random_state=seed
    )

    train_val_idx, test_idx = next(
        splitter_1.split(X, groups=groups)
    )

    train_val = X.iloc[train_val_idx]

    # Second split: train and validation
    splitter_2 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.1765,
        random_state=seed + 10000
    )

    train_idx, val_idx = next(
        splitter_2.split(
            train_val,
            groups=train_val["question_focus"]
        )
    )

    # Calculate actual proportions
    sizes = [
        len(train_idx),
        len(val_idx),
        len(test_idx)
    ]

    proportions = [size / len(X) for size in sizes]

    # Calculate distance from target proportions
    score = sum(
        (actual - desired) ** 2
        for actual, desired in zip(proportions, target)
    )

    if score < best_score:
        best_score = score
        best_split = (
            train_val.iloc[train_idx].copy(),
            train_val.iloc[val_idx].copy(),
            X.iloc[test_idx].copy()
        )


# Save optimized datasets
train_data, val_data, test_data = best_split

train_data.reset_index(drop=True, inplace=True)
val_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)


# Display results
print("========== OPTIMIZED SPLIT RESULTS ==========")

for name, dataset in [
    ("TRAIN", train_data),
    ("VALIDATION", val_data),
    ("TEST", test_data)
]:
    print(f"\n{name}")
    print("Examples:", len(dataset))
    print("Percentage:", round(len(dataset) / len(X) * 100, 2), "%")
    print("Unique topics:", dataset["question_focus"].nunique())


# Validate group separation
train_groups = set(train_data["question_focus"])
val_groups = set(val_data["question_focus"])
test_groups = set(test_data["question_focus"])

print("\n========== DATA LEAKAGE CHECK ==========")

print("Train / Validation:", len(train_groups & val_groups))
print("Train / Test:", len(train_groups & test_groups))
print("Validation / Test:", len(val_groups & test_groups))

print("\n========== FINAL CHECK ==========")

print("Total examples:", len(train_data) + len(val_data) + len(test_data))
print(
    "All groups separated:",
    not (train_groups & val_groups
         or train_groups & test_groups
         or val_groups & test_groups)
)

========== OPTIMIZED SPLIT RESULTS ==========

TRAIN
Examples: 254
Percentage: 69.78 %
Unique topics: 44

VALIDATION
Examples: 55
Percentage: 15.11 %
Unique topics: 10

TEST
Examples: 55
Percentage: 15.11 %
Unique topics: 10

========== DATA LEAKAGE CHECK ==========
Train / Validation: 0
Train / Test: 0
Validation / Test: 0

========== FINAL CHECK ==========
Total examples: 364
All groups separated: True


In [20]:
# ==========================================
# Cell 20 - Split Composition Audit
# ==========================================

datasets = {
    "TRAIN": train_data,
    "VALIDATION": val_data,
    "TEST": test_data
}

for name, dataset in datasets.items():

    print(f"\n========== {name} ==========")

    print("\nTopic groups:")
    print(
        dataset.groupby("topic_group")
        .size()
        .to_string()
    )

    print("\nMedical topics:")
    print(
        dataset.groupby("question_focus")
        .size()
        .sort_values(ascending=False)
        .to_string()
    )

    print("\nSources:")
    print(
        dataset["document_source"]
        .value_counts()
        .to_string()
    )


# Final integrity checks
print("\n========== INTEGRITY CHECKS ==========")

for name, dataset in datasets.items():
    print(f"\n{name}")
    print("Missing questions:", dataset["question"].isna().sum())
    print("Missing answers:", dataset["answer"].isna().sum())
    print("Duplicate Q&A pairs:", dataset.duplicated(
        subset=["question", "answer"]
    ).sum())


========== TRAIN ==========

Topic groups:
topic_group
main             247
supplementary      7

Medical topics:
question_focus
Stroke                                                                  35
High Blood Cholesterol                                                  28
High Blood Pressure                                                     27
Renal Artery Stenosis                                                   10
Carotid Artery Disease                                                   8
Diabetic Heart Disease                                                   8
High Blood Pressure and Kidney Disease                                   7
Heart Valve Disease                                                      7
Coronary Microvascular Disease                                           7
Arrhythmia                                                               7
Coronary Heart Disease                                                   7
Broken Heart Syndrome                        

In [21]:
# ==========================================
# Cell 21 - JSONL Export
# ==========================================

import os
import json

# Create output directory
output_dir = "/kaggle/working/cardioagent_dataset"
os.makedirs(output_dir, exist_ok=True)

# Convert dataframe into conversation records
def create_records(dataset):

    records = []

    for _, row in dataset.iterrows():

        record = {
            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": row["question"]
                },
                {
                    "role": "assistant",
                    "content": row["answer"]
                }
            ],
            "metadata": {
                "question_focus": row["question_focus"],
                "document_source": row["document_source"],
                "document_id": row["document_id"],
                "topic_group": row["topic_group"]
            }
        }

        records.append(record)

    return records


# Prepare the three datasets
datasets = {
    "train": train_data,
    "validation": val_data,
    "test": test_data
}

# Export each split
for split_name, dataset in datasets.items():

    records = create_records(dataset)

    file_path = os.path.join(
        output_dir,
        f"{split_name}.jsonl"
    )

    with open(file_path, "w", encoding="utf-8") as file:

        for record in records:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    print(f"{split_name}.jsonl exported: {len(records)} examples")


print("\n========== EXPORTED FILES ==========")

print(os.listdir(output_dir))

train.jsonl exported: 254 examples
validation.jsonl exported: 55 examples
test.jsonl exported: 55 examples

========== EXPORTED FILES ==========
['validation.jsonl', 'train.jsonl', 'test.jsonl']


In [22]:
# ==========================================
# Cell 22 - JSONL Validation
# ==========================================

for split_name, dataset in datasets.items():

    file_path = os.path.join(
        output_dir,
        f"{split_name}.jsonl"
    )

    with open(file_path, "r", encoding="utf-8") as file:
        loaded_records = [
            json.loads(line)
            for line in file
        ]

    print(f"\n========== {split_name.upper()} ==========")

    print("Expected examples:", len(dataset))
    print("Loaded examples:", len(loaded_records))

    print(
        "All records have 3 messages:",
        all(len(item["messages"]) == 3 for item in loaded_records)
    )

    print(
        "All records have metadata:",
        all("metadata" in item for item in loaded_records)
    )

    print(
        "All roles valid:",
        all(
            [msg["role"] for msg in item["messages"]]
            == ["system", "user", "assistant"]
            for item in loaded_records
        )
    )

    print(
        "Count matches:",
        len(loaded_records) == len(dataset)
    )


========== TRAIN ==========
Expected examples: 254
Loaded examples: 254
All records have 3 messages: True
All records have metadata: True
All roles valid: True
Count matches: True

========== VALIDATION ==========
Expected examples: 55
Loaded examples: 55
All records have 3 messages: True
All records have metadata: True
All roles valid: True
Count matches: True

========== TEST ==========
Expected examples: 55
Loaded examples: 55
All records have 3 messages: True
All records have metadata: True
All roles valid: True
Count matches: True
